# Deteksi TB — Eksperimen Multi-Seed 3 Seed (mean ± std + uji signifikansi)
### Untuk memperkuat klaim statistik paper J. ICT Res. Appl.

**Tujuan:** menjalankan tiap arsitektur dengan beberapa seed berbeda, lalu melaporkan
**rata-rata ± standar deviasi** dan **uji signifikansi** antar model. Ini membuktikan
hasil bukan kebetulan dari satu split data.

**PENTING — soal waktu & ketahanan:**
- 5 model × 3 seed = 15 training penuh. Di Colab T4 gratis ini cukup lama (~5 jam) dan bisa terputus.
- Notebook ini menyimpan hasil tiap (model, seed) ke Drive secara bertahap.
- Jika sesi putus, jalankan ulang dari atas: kombinasi yang sudah selesai akan DILEWATI otomatis.
- Untuk uji cepat dulu, kecilkan `SEEDS` jadi [42] dan `MODELS_TO_RUN` jadi 2 model, lalu perbesar.

**WAJIB:** aktifkan GPU (Runtime > Change runtime type > T4 GPU).


## 1. Mount Drive (untuk simpan hasil bertahap) & cek GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
RESULT_DIR = "/content/drive/MyDrive/tb_multiseed_3seed"
os.makedirs(RESULT_DIR, exist_ok=True)
print("Hasil akan disimpan di:", RESULT_DIR)
!nvidia-smi -L

## 2. Unduh dataset & tata folder

In [ ]:
import os
if not os.path.isdir("data/main"):
    !pip install -q kaggle
    !kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset
    !unzip -q -o tuberculosis-tb-chest-xray-dataset.zip -d data_raw
    import shutil
    src = "data_raw/TB_Chest_Radiography_Database"
    os.makedirs("data/main", exist_ok=True)
    for cls in ["Normal", "Tuberculosis"]:
        if not os.path.isdir(f"data/main/{cls}"):
            shutil.move(f"{src}/{cls}", f"data/main/{cls}")
print("Normal:", len(os.listdir("data/main/Normal")), "| TB:", len(os.listdir("data/main/Tuberculosis")))

## 3. Konfigurasi

In [ ]:
import os, time, json, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import (
    ResNet50, EfficientNetB0, MobileNetV2, DenseNet121, VGG16,
    resnet50, efficientnet, mobilenet_v2, densenet, vgg16)
from sklearn.metrics import (roc_curve, auc, accuracy_score,
    precision_recall_fscore_support, f1_score)

IMG_SIZE = (224, 224); BATCH_SIZE = 32
EPOCHS_HEAD, EPOCHS_FT = 8, 12
LR_HEAD, LR_FT = 1e-3, 1e-5
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15
DATA_MAIN = "data/main"
RESULT_DIR = "/content/drive/MyDrive/tb_multiseed_3seed"

# Seed berbeda untuk uji kestabilan. Untuk uji cepat: [42, 1]
SEEDS = [42, 1, 7]
MODELS_TO_RUN = ["ResNet50","EfficientNetB0","MobileNetV2","DenseNet121","VGG16"]

ALL_MODELS = {
    "ResNet50":       (ResNet50,       resnet50.preprocess_input),
    "EfficientNetB0": (EfficientNetB0, efficientnet.preprocess_input),
    "MobileNetV2":    (MobileNetV2,    mobilenet_v2.preprocess_input),
    "DenseNet121":    (DenseNet121,    densenet.preprocess_input),
    "VGG16":          (VGG16,          vgg16.preprocess_input),
}
print("TF:", tf.__version__, "| Seeds:", SEEDS)

## 4. Fungsi

In [ ]:
def build_datasets(preprocess, seed):
    full = tf.keras.utils.image_dataset_from_directory(
        DATA_MAIN, labels="inferred", label_mode="binary",
        class_names=["Normal","Tuberculosis"], image_size=IMG_SIZE,
        batch_size=None, shuffle=True, seed=seed)   # seed mengubah split & shuffle
    n = full.cardinality().numpy()
    n_test, n_val = int(n*TEST_SPLIT), int(n*VAL_SPLIT)
    test_ds = full.take(n_test); rest = full.skip(n_test)
    val_ds = rest.take(n_val);   train_ds = rest.skip(n_val)
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
        layers.RandomZoom(0.10), layers.RandomContrast(0.10)])
    AT = tf.data.AUTOTUNE
    def prep(img, lab, tr):
        img = preprocess(tf.cast(img, tf.float32))
        if tr: img = aug(img, training=True)
        return img, lab
    train_ds = train_ds.map(lambda x,y: prep(x,y,True), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    val_ds   = val_ds.map(lambda x,y: prep(x,y,False), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    test_ds  = test_ds.map(lambda x,y: prep(x,y,False), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    return train_ds, val_ds, test_ds

def build_model(builder):
    base = builder(include_top=False, weights="imagenet", input_shape=IMG_SIZE+(3,), pooling="avg")
    base.trainable = False
    inp = tf.keras.Input(shape=IMG_SIZE+(3,))
    x = base(inp, training=False)
    x = layers.Dropout(0.3)(x); x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x); out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out), base

def get_probs(model, ds):
    yt, yp = [], []
    for xb, yb in ds:
        yp.extend(model.predict(xb, verbose=0).ravel().tolist())
        yt.extend(yb.numpy().ravel().tolist())
    return np.array(yt), np.array(yp)

def best_threshold(yt, yp):
    grid = np.linspace(0.05, 0.95, 91)
    return float(grid[int(np.argmax([f1_score(yt,(yp>=t).astype(int),zero_division=0) for t in grid]))])

def metrics_at(yt, yp, thr):
    pred = (yp>=thr).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yt, pred, average="binary", zero_division=0)
    fpr, tpr, _ = roc_curve(yt, yp)
    return {"accuracy":accuracy_score(yt,pred), "precision":pr, "recall":rc, "f1":f1, "auc":auc(fpr,tpr)}

def run_one(name, seed):
    """Latih 1 model dgn 1 seed; kembalikan metrik tuned. Hasil di-cache ke Drive (atomik)."""
    cache = f"{RESULT_DIR}/{name}_seed{seed}.json"
    if os.path.exists(cache):
        try:
            with open(cache) as f: return json.load(f)   # sudah selesai & valid → lewati
        except Exception:
            os.remove(cache)  # file korup (putus saat nulis sebelumnya) → ulang
    tf.random.set_seed(seed); np.random.seed(seed)
    builder, preprocess = ALL_MODELS[name]
    train_ds, val_ds, test_ds = build_datasets(preprocess, seed)
    model, base = build_model(builder)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    cw = {0:1.0, 1:3500/700}
    model.compile(optimizers.Adam(LR_HEAD), "binary_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=[es], class_weight=cw, verbose=0)
    base.trainable = True
    for L in base.layers[:-30]: L.trainable = False
    model.compile(optimizers.Adam(LR_FT), "binary_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=[es], class_weight=cw, verbose=0)
    yv, pv = get_probs(model, val_ds); thr = best_threshold(yv, pv)
    yt, yp = get_probs(model, test_ds); m = metrics_at(yt, yp, thr)
    m = {k: float(round(v,4)) for k,v in m.items()}; m["threshold"] = round(thr,3)
    # --- penyimpanan ATOMIK: tulis ke .tmp lalu rename, agar cache tak pernah korup ---
    tmp = cache + ".tmp"
    with open(tmp, "w") as f: json.dump(m, f)
    os.replace(tmp, cache)   # rename atomik
    tf.keras.backend.clear_session()
    return m

def done_count():
    """Hitung berapa kombinasi sudah tersimpan di Drive."""
    done = 0
    for nm in MODELS_TO_RUN:
        for sd in SEEDS:
            if os.path.exists(f"{RESULT_DIR}/{nm}_seed{sd}.json"): done += 1
    return done

print("Fungsi siap.")

## 5. Jalankan semua kombinasi (model × seed)
Tiap kombinasi yang selesai langsung tersimpan ke Drive. **Aman bila terputus** — jalankan ulang sel ini (atau seluruh notebook) untuk melanjutkan; yang sudah selesai dilewati otomatis.

In [ ]:
total = len(MODELS_TO_RUN) * len(SEEDS)
print(f"Total kombinasi: {total} | sudah selesai sebelumnya: {done_count()}")
print("="*60)
times = []
for name in MODELS_TO_RUN:
    for seed in SEEDS:
        already = os.path.exists(f"{RESULT_DIR}/{name}_seed{seed}.json")
        t0 = time.time()
        m = run_one(name, seed)
        dt = time.time() - t0
        tag = "(cache)" if already else f"({dt:.0f}s)"
        if not already: times.append(dt)
        d = done_count()
        eta = (np.mean(times) * (total - d)) / 60 if times else 0
        print(f"[{d}/{total}] {name} seed={seed}: acc={m['accuracy']:.4f} "
              f"recall={m['recall']:.4f} f1={m['f1']:.4f} auc={m['auc']:.4f} {tag}"
              + (f" | sisa ~{eta:.0f} min" if times and d<total else ""))
print("="*60)
print(f"Selesai: {done_count()}/{total} kombinasi.")

### (Opsional) Keepalive — cegah Colab putus karena idle
Jalankan sel ini di tab terpisah / sebelum training panjang. Ia hanya mencetak waktu tiap menit agar sesi tidak dianggap menganggur. Hentikan manual saat training selesai.

In [ ]:
# OPSIONAL — jalankan hanya jika sesi sering putus karena idle.
# import time
# from datetime import datetime
# for i in range(300):   # ~5 jam
#     print("keepalive", datetime.now().strftime("%H:%M:%S")); time.sleep(60)

## 6. Ringkasan mean ± std per model
Sel ini membangun ulang `df` dari **semua file hasil di Drive**, jadi tetap berfungsi penuh walau notebook dijalankan di sesi baru setelah terputus (selama file cache ada di Drive). Bisa dijalankan dengan hasil parsial sekalipun.

In [ ]:
# Bangun ulang df dari file-file JSON di Drive (tahan restart / hasil parsial)
import glob, json
rows = []
for fp in glob.glob(f"{RESULT_DIR}/*_seed*.json"):
    base = os.path.basename(fp)[:-5]            # buang .json
    name, seedpart = base.rsplit("_seed", 1)
    try:
        with open(fp) as f: m = json.load(f)
    except Exception:
        continue
    rows.append({"model":name, "seed":int(seedpart), **m})
df = pd.DataFrame(rows).sort_values(["model","seed"]).reset_index(drop=True)
df.to_csv(f"{RESULT_DIR}/all_runs.csv", index=False)
print(f"Memuat {len(df)} run dari Drive.")
n_per = df.groupby("model")["seed"].count()
print("Jumlah seed selesai per model:"); print(n_per.to_string())

summary = pd.DataFrame(index=sorted(df["model"].unique()))
for metric in ["accuracy","precision","recall","f1","auc"]:
    summary[metric] = [f"{df[df.model==m][metric].mean():.4f} ± {df[df.model==m][metric].std():.4f}"
                       for m in summary.index]
summary.to_csv(f"{RESULT_DIR}/summary_mean_std.csv")
print("\nRingkasan mean ± std:")
display(summary)

## 7. Uji signifikansi statistik
Membandingkan model dengan **F1 rata-rata tertinggi** melawan model lain menggunakan
**paired t-test** dan **Wilcoxon signed-rank** (berpasangan per seed). p < 0.05 → perbedaan signifikan.

In [ ]:
from scipy import stats
# tentukan model terbaik berdasar mean F1
mean_f1 = df.groupby("model")["f1"].mean().sort_values(ascending=False)
best = mean_f1.index[0]
print(f"Model terbaik (mean F1): {best} = {mean_f1.iloc[0]:.4f}\n")

sig_rows = []
for other in mean_f1.index[1:]:
    a = df[df.model==best].sort_values("seed")["f1"].values
    b = df[df.model==other].sort_values("seed")["f1"].values
    if len(a)==len(b) and len(a)>=2:
        t_p = stats.ttest_rel(a, b).pvalue
        try: w_p = stats.wilcoxon(a, b).pvalue
        except Exception: w_p = float("nan")
        sig_rows.append({"comparison":f"{best} vs {other}",
                         "mean_F1_diff":round(a.mean()-b.mean(),4),
                         "t_test_p":round(t_p,4),
                         "wilcoxon_p":round(w_p,4),
                         "significant_(p<0.05)": "YES" if t_p<0.05 else "no"})
sig = pd.DataFrame(sig_rows)
sig.to_csv(f"{RESULT_DIR}/significance_tests.csv", index=False)
display(sig)

## 8. Tabel siap-paper (format ringkas)
Salin tabel ini ke manuskrip menggantikan tabel single-run.

In [ ]:
paper_tbl = summary.copy()
paper_tbl.columns = ["Accuracy","Precision","Recall (TB)","F1","AUC"]
print("Tabel untuk paper (mean ± std, n =", len(SEEDS), "seeds):")
display(paper_tbl)
print("\nFile tersimpan di Drive:")
print(" -", f"{RESULT_DIR}/all_runs.csv          (semua run mentah)")
print(" -", f"{RESULT_DIR}/summary_mean_std.csv  (ringkasan mean±std)")
print(" -", f"{RESULT_DIR}/significance_tests.csv (uji signifikansi)")

---
### Cara membaca untuk paper

1. **Tabel mean ± std** menggantikan tabel single-run. Contoh klaim: *"EfficientNetB0 mencapai F1 0.95 ± 0.01 lintas 5 seed."* std kecil = stabil.
2. **Uji signifikansi:** jika p < 0.05, Anda boleh menulis *"model A secara signifikan mengungguli model B (paired t-test, p < 0.05)."* Jika p ≥ 0.05, jujur tulis bahwa selisihnya tidak signifikan — ini temuan yang valid.
3. **Jujur soal hasil:** kalau setelah multi-seed ternyata beberapa model setara secara statistik, laporkan apa adanya. Itu justru menambah kredibilitas, bukan mengurangi.

**Catatan:** 3 seed sudah cukup umum diterima untuk menunjukkan kestabilan dan menjalankan uji signifikansi. Jika sempat, Anda bisa menambah seed (mis. [42,1,7,123,2024]) untuk klaim lebih kuat.
